# 集成学习

在深入这两个算法之前，必须先理解集成学习 (Ensemble Learning)。它的核心思想是：“三个臭皮匠，顶个诸葛亮”。通过构建并结合多个“弱学习器”（通常是单棵决策树）来构建一个“强学习器”。

集成学习主要分为两大派系：

Bagging（套袋法）：各个弱学习器之间并行独立生成，最后通过投票或取平均的方式结合。随机森林是其典型代表。

Boosting（提升法）：各个弱学习器之间串行存在依赖关系，每一棵新树都在努力修正前面所有树犯下的错误。GBDT是其典型代表。

# 随机森林 (Random Forest)

概念与原理随机森林是一种基于 Bagging 策略的集成算法。它的基学习器是 CART 决策树。它的“随机”体现在两个核心维度（双重随机性）：

- 样本随机性：使用 Bootstrap（自助采样法），从原始训练集中有放回地随机抽取 $N$ 个样本进行每棵树的训练。
- 特征随机性：在决策树分裂节点时，不从全部 $M$ 个特征中选最优，而是随机选择 $k$ 个特征（通常 $k = \sqrt{M}$），再从中选择最优分裂点。

这种双重随机性极大地增加了基学习器之间的差异性（多样性），从而让集成后的模型具备极强的泛化能力。

## 数学理论

随机森林的数学本质是通过降低方差（Variance）来提升泛化能力。

对于单棵决策树，往往容易过拟合，这意味着它的方差很大。假设我们有 $n$ 个独立同分布的单棵树，每棵树的方差为 $\sigma^2$。如果我们对它们取平均：

$$\text{Var}\left(\frac{1}{n}\sum_{i=1}^n X_i\right) = \frac{\sigma^2}{n}$$

而在随机森林中，由于特征和样本的随机性，树与树之间并不是完全独立的，假设它们之间的相关系数是 $\rho$，那么集成后的方差为：

$$\text{Var} = \rho \sigma^2 + \frac{1-\rho}{n}\sigma^2$$

随着树的数量 $n$ 增加，第二项趋近于 0，而第一项 $\rho \sigma^2$ 决定了泛化误差的下界。随机森林通过引入特征随机性，极大地降低了树之间的相关性 $\rho$，从而显著降低了总方差。

## 应用领域、特点与优缺点

应用领域：金融风控（信贷准入、欺诈检测）、生物信息学（基因筛选）、客户流失预测等。

优点：

- 极不易过拟合：双重随机性带来了极强的鲁棒性。

- 抗噪能力强：对缺失值、异常值不敏感。
- 天然支持并行化：树与树之间相互独立，训练速度快。
- 能输出特征重要性 (Feature Importance)。

能输出特征重要性 (Feature Importance)。

缺点：

- 对于高维稀疏的文本数据，表现不如线性模型。

- 预测速度相比单棵树较慢（需要遍历所有树）。

# 梯度提升树 (GBDT)

## 概念与原理

GBDT是一种基于 Boosting 策略的迭代回归树算法。注意：无论处理的是分类还是回归，GBDT 的基学习器都必须是回归树 (CART Regression Tree)。
它的核心思想是：沿梯度的反方向建立新树，来拟合当前模型的残差（Residuals）。想象一下你在打高尔夫球，第一杆打了 200 米（离洞还差 30 米），第二杆你就需要针对这“剩下的 30 米（残差）”来打，而不是重新从起点打。GBDT 就是这样，第二棵树去拟合第一棵树预测剩下的残差，第三棵树去拟合前两棵树相加后的残差，以此类推。

## 数学理论

假设我们要最小化一个损失函数 $L(y, f(x))$。在第 $m$ 步，我们的模型是：

$$f_m(x) = f_{m-1}(x) + \gamma_m h_m(x)$$

其中 $h_m(x)$ 是新加入的决策树。如何让损失函数最小？根据泰勒一阶展开，我们要让新树 $h_m(x)$ 去拟合损失函数的负梯度（在平方损失下，负梯度恰好等于残差）：

$$r_{im} = -\left[ \frac{\partial L(y_i, f(x_i))}{\partial f(x_i)} \right]_{f(x) = f_{m-1}(x)}$$

每一轮迭代，我们都计算当前模型在所有样本上的负梯度 $r_{im}$，然后训练一棵新树 $h_m(x)$ 去拟合这个负梯度，最后通过线性和（通常伴随一个学习率/步长 $\nu$）累加到模型中：$$f_m(x) = f_{m-1}(x) + \nu \cdot \gamma_m h_m(x)$$

## 应用领域、特点与优缺点应用领域：

- 搜索广告点击率预测 (CTR)、搜索排序 (LambdaMART)、推荐系统、各种 Kaggle 比赛。

__优点：__

- 预测精度高：工业界公认的杀手级算法，表达能力极强。
- 能灵活处理各种类型的损失函数（通过自定义梯度）。


__缺点：__

- 难以并行化：因为树之间是串行依赖的（后一棵树依赖前一棵树的结果）。
- 相比 RF，GBDT 调参难度大，更容易过拟合（如果树太深或迭代次数太多）。

In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. 模拟生成结构化数据集 (1000个样本，20个特征)
X, y = make_classification(n_samples=1000, n_features=20,
                           n_informative=15, n_redundant=5,
                           random_state=42)

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"训练集规模: {X_train.shape}, 测试集规模: {X_test.shape}")
print("-" * 50)

# 2. 随机森林 (Random Forest) 应用接口
# n_estimators: 树的棵树
# max_depth: 树的最大深度（防止单棵树过拟合）
# max_features: 寻找最优分裂时考虑的特征数
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, max_features='sqrt', random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
print(f"【随机森林】测试集准确率: {accuracy_score(y_test, rf_preds):.4f}")
# 输出特征重要性 (前5个)
print("【随机森林】前5个特征重要性:", rf_model.feature_importances_[:5])
print("-" * 50)

# 3. 梯度提升树 (GBDT) 应用接口
# learning_rate: 学习率/步长（收缩率）
# n_estimators: 迭代次数（树的棵树）
gbdt_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gbdt_model.fit(X_train, y_train)

gbdt_preds = gbdt_model.predict(X_test)
print(f"【GBDT】测试集准确率: {accuracy_score(y_test, gbdt_preds):.4f}")
print("-" * 50)

训练集规模: (800, 20), 测试集规模: (200, 20)
--------------------------------------------------
【随机森林】测试集准确率: 0.9000
【随机森林】前5个特征重要性: [0.05169048 0.04649107 0.08347786 0.03995216 0.04972149]
--------------------------------------------------
【GBDT】测试集准确率: 0.9100
--------------------------------------------------


'\n- RF 调参重点：n_estimators (通常越多越好，边际效益递减), max_features, min_samples_leaf。\n- GBDT 调参重点：learning_rate 和 n_estimators 具有此消彼长的关系。通常先固定一个较小的 learning_rate (如 0.05 或 0.1)，\n  然后通过交叉验证寻找最佳的 n_estimators；另外控制 max_depth (通常 3-8 即可，不需要太深)。\n'

### 核心调参老兵建议：

- RF 调参重点：n_estimators (通常越多越好，边际效益递减), max_features, min_samples_leaf。
- GBDT 调参重点：learning_rate 和 n_estimators 具有此消彼长的关系。通常先固定一个较小的 learning_rate (如 0.05 或 0.1)，
  然后通过交叉验证寻找最佳的 n_estimators；另外控制 max_depth (通常 3-8 即可，不需要太深)。

# XGBoost

今天我们要聊的 XGBoost (Extreme Gradient Boosting)，是陈天奇大佬在 2014 年提出的。在深度学习统治大模型时代的今天，XGBoost 依然是工业界处理结构化数据、各种 Kaggle/天池比赛中雷打不动的“大杀器”。

简单来说，XGBoost 就是把 GBDT 推向了数学和工程实现的“极端（Extreme）”。

## 概念与基本原理

还记得 GBDT 是怎么优化的吗？GBDT 沿梯度的反方向（一阶导数）去拟合残差。
而 XGBoost 在此基础上做了两大核心升级：

数学上：引入了二阶泰勒展开，同时在目标函数中加入了正则化项（惩罚树的复杂度）。

工程上：设计了高度优化的并行结构、缺失值自适应分裂、稀疏矩阵处理以及内存缓存优化。

## 数学理论

### 目标函数与正则化

在第 $t$ 轮，我们要加入一棵新树 $f_t(x)$。XGBoost 的目标函数（Objective Function）定义为：

$$\mathcal{L}^{(t)} = \sum_{i=1}^n l\left(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\right) + \Omega(f_t)$$

这里的 $\Omega(f_t)$ 是正则化项，用来控制树的复杂度，防止过拟合：

$$\Omega(f_t) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2$$

- $T$ 是这棵树的叶子节点总数（叶子越多，模型越复杂，给予惩罚 $\gamma$）。
- $w_j$ 是第 $j$ 个叶子节点的预测得分（权重）（权重绝对值越大，越容易过拟合，给予 $L_2$ 惩罚 $\lambda$）。

### 二阶泰勒展开

GBDT 只用了一阶导数，而 XGBoost 对损失函数进行了二阶泰勒展开。
令一阶导数（梯度）为 $g_i = \partial_{\hat{y}^{(t-1)}} l(y_i, \hat{y}_i^{(t-1)})$，二阶导数（Hessian矩阵）为
$h_i = \partial^2_{\hat{y}^{(t-1)}} l(y_i, \hat{y}_i^{(t-1)})$。

目标函数可以近似为：

$$\mathcal{L}^{(t)} \approx \sum_{i=1}^n \left[ l(y_i, \hat{y}_i^{(t-1)}) + g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2$$

移除常数项 $l(y_i, \hat{y}_i^{(t-1)})$（因为它是前 $t-1$ 轮算出来的，对当前优化无影响），并将样本遍历转换为叶子节点遍历。假设 $I_j$ 是被分到第 $j$ 个叶子节点上的样本集合：

$$\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^T \left[ \left(\sum_{i \in I_j} g_i\right) w_j + \frac{1}{2}\left(\sum_{i \in I_j} h_i + \lambda\right) w_j^2 \right] + \gamma T$$

为了公式清爽，我们定义大 $G$ 和大 $H$：

- $G_j = \sum_{i \in I_j} g_i$ （叶子 $j$ 上所有样本的一阶导数和）
- $H_j = \sum_{i \in I_j} h_i$ （叶子 $j$ 上所有样本的二阶导数和）

$$\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^T \left[ G_j w_j + \frac{1}{2}(H_j + \lambda) w_j^2 \right] + \gamma T$$

### 求解最优叶子权重与最优分数

现在这是一个关于 $w_j$ 的一元二次方程！对 $w_j$ 求导并令其为 0，我们可以直接闭式解出最优叶子权重 $w_j^*$：

$$w_j^* = -\frac{G_j}{H_j + \lambda}$$

把 $w_j^*$ 代回目标函数，得到最优目标函数值（也就是树的结构得分评分标准，Score）：

$$\mathcal{L}^{(t)*} = -\frac{1}{2} \sum_{j=1}^T \frac{G_j^2}{H_j + \lambda} + \gamma T$$

这个分数越小越好，它代表了当前树结构下模型的最大拟合能力。

### 分裂增益（Gain）

当我们在选哪个特征、哪个切分点来把一个叶子节点分裂成左右两个新叶子时，增益计算公式为：

$$\text{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \right] - \gamma$$

通俗解释：[左子树分数] + [右子树分数] - [不分裂时的原分数] - [多出一个叶子节点的惩罚项]。如果 $\text{Gain} > 0$，说明分裂能带来收益。

## 核心特点与工程优化

为什么 XGBoost 比传统 GBDT 快这么多？

- 精确贪心算法与近似算法：不仅支持遍历所有可能切分点的精确模式，还支持基于分位数（Quantile Sketch）的近似切分算法，在大数据量下快得飞起。
- 稀疏矩阵与缺失值处理（Sparsity-aware Split Finding）：如果数据有缺失，XGBoost 会分别尝试把缺失样本分到左子树和右子树，看哪边增益大，从而为缺失值自动学出一个默认方向。
- 列块并行（Column Block）：在训练前，对特征数据进行预排序并存储为 Block 结构。在寻找切分点时，可以并行计算不同特征的增益。注意：它是特征间并行，而不是树间并行。
- 缓存优化（Cache-aware Access）：预先申请线程连续的内部缓冲区，提升 CPU 缓存命中率。

## 应用领域与优缺点

| 维度   | 说明                                                                                                  |
| ---- | --------------------------------------------------------------------------------------------------- |
| 应用领域 | 推荐系统排序、金融风控评分卡、电商转化率（CVR）预测、传统结构化表格数据的分类与回归。                                                        |
| 优点   | 1. 精度极高，二阶导数比一阶导数更精准。<br>2. 自带 L1/L2 正则化，相比 GBDT 更不易过拟合。<br>3. 工业级工程加速，支持分布式、GPU 训练。<br>4. 自动处理缺失值。 |
| 缺点   | 1. 空间复杂度较高，预排序需要消耗大量内存。<br>2. 对时序数据或非表格数据（图像、语音等）的表达能力有限。                                           |


In [4]:
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score

# 1. 构建模拟数据集
X, y = make_classification(n_samples=2000, n_features=20, n_informative=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. 初始化 XGBoost 分类器
# tree_method='hist' 启用直方图优化（类似LightGBM，速度极快）
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.08,
    max_depth=5,
    subsample=0.8,  # 样本采样率
    colsample_bytree=0.8,  # 特征采样率（类似随机森林）
    reg_alpha=0.1,  # L1 正则化系数
    reg_lambda=1.0,  # L2 正则化系数
    tree_method='hist',  # 现代加速首选
    random_state=42
)

# 3. 模型训练
xgb_model.fit(X_train, y_train)

# 4. 预测评估
preds = xgb_model.predict(X_test)
pred_probs = xgb_model.predict_proba(X_test)[:, 1]

print(f"【XGBoost】测试集准确率: {accuracy_score(y_test, preds):.4f}")
print(f"【XGBoost】测试集LogLoss: {log_loss(y_test, pred_probs):.4f}")

# 5. 查看特征重要性
importance = xgb_model.feature_importances_
print("前3个特征的相对重要性权重:", importance[:3])

【XGBoost】测试集准确率: 0.9350
【XGBoost】测试集LogLoss: 0.2028
前3个特征的相对重要性权重: [0.04500053 0.05353142 0.01742956]


# LightGBM

如果说 XGBoost 是把传统 GBDT 的精度推向了极致，那么 LightGBM 就是在 XGBoost 的基础上，把速度和内存利用率推向了另一个巅峰。

在处理超大规模数据集时，传统的 XGBoost 常常会因为内存吃紧、训练时间过长而被戏称为“内存粉碎机”。而 LightGBM 的出现，完美解决了这个痛点，它的核心字眼就是：快、轻、省

## 概念与核心生长机制

LightGBM 在策略上最大的颠覆，在于改变了树的生长机制。

Level-wise（层级生长） vs Leaf-wise（叶子生长）

- XGBoost 默认采用 Level-wise：对同一层的所有叶子节点一视同仁，强行进行分裂。这种方式比较稳妥，容易通过控制树深（max_depth）来防止过拟合，但它比较“盲目”，因为很多叶子节点的分类增益非常小，强行分裂会带来不必要的计算开销。

- LightGBM 采用 Leaf-wise：它是一种更高效的贪心策略。每次分裂时，它会遍历当前所有叶子节点，只选择分裂增益最大（能让损失函数下降最多）的那一个叶子节点进行分裂，如此循环。

Leaf-wise 能够学到比 Level-wise 更深、更复杂的树结构，从而获得更高的精度。但它的代价是极易过拟合。因此，使用 LightGBM 时，必须通过严格限制叶子总数（num_leaves）和最大深度（max_depth）来拉住这匹脱缰的野马。

## 数学与算法创新

LightGBM 能够做到又快又省，数学与工程层面的两项创新功不可没：GOSS 和 EFB。

### GOSS（基于梯度的单侧采样）

在 Boosting 算法中，每个样本的梯度大小代表了它的训练程度。梯度越大，说明模型预测越不准，该样本的误差越大、越“缺练”。

统的 GBDT 在每一轮都需要遍历所有样本来计算增益，而 GOSS 的核心思想是：扔掉一部分已经训练得很好的小梯度样本，只保留大梯度样本，但在计算增益时给小梯度样本进行补偿。

#### 具体数学实现步骤

- 将输入样本按照梯度的绝对值从大到小排序。
- 选出前 $a \times 100\%$ 的大梯度样本，构成集合 $A$。
- 从剩下的 $(1-a) \times 100\%$ 的小梯度样本中，随机抽取 $b \times 100\%$ 的样本，构成集合 $B$
- 在计算分裂增益时，为了不改变原始数据的分布，LightGBM 给集合 $B$ 中的小梯度样本乘上了一个权重补偿系数：$\frac{1-a}{b}$。

通过这种方式，模型既能将注意力集中在未训练好的大误差样本上，又保证了整体数据分布的统计一致性。

### EFB（互斥特征捆绑）

高维数据往往非常稀疏（比如 One-hot 编码后的特征），很多特征几乎不会同时出现非零值（它们是“互斥”的）。

EFB 的作用就是把这些互斥的特征“捆绑”成一个新的复合特征（特征图谱）。

> 做法：将不同的特征赋予不同的偏置（Offset）。例如，特征 A 的取值范围是 $[0, 10]$，特征 B 的取值范围是 $[0, 20]$。为了把它们绑在一起又不产生冲突，我们可以让特征 B 的值加上
>  10，变成 $[10, 30]$。这样，一个合并特征就能完美表达原本两个特征的信息，直接将特征维度降了下来。

## 工程优化：直方图（Histogram）算法

XGBoost 默认需要对特征值进行预排序（Pre-sorted），这需要消耗双倍的内存（既要存特征值，又要存排序后的索引）。

LightGBM 彻底抛弃了预排序，采用了直方图算法：

- 将连续的浮点特征值离散化成 $K$ 个整数桶（Bins，通常设为 256）。
- 在训练时，不需要再遍历所有样本，只需要遍历这 256 个 Bin 即可。计算复杂度直接从 $O(\#\text{data})$ 降到了 $O(\#\text{bins})$。
- 直方图做差加速：一个叶子节点的直方图，可以通过它父亲节点的直方图减去它兄弟节点的直方图直接得到。由于这个数学特性，LightGBM 构建一个新叶子的直方图只需消耗一半的计算量。

## 特点与优缺点对比

__优点：__

- 速度极快：通常比传统 XGBoost 快数倍。
- 内存消耗极低：直方图算法大幅压缩了特征存储开销，且不需要预排序。
- 准确率高：Leaf-wise 生长策略往往能找到比 Level-wise 更优的树结构。
- 原生支持类别特征 (Categorical Features)：不需要手动做 One-hot，直接在算法底层进行类别划分。

__缺点：__

- 在小数据集上极易过拟合。
- Leaf-wise 相比 Level-wise 对噪声更敏感。

In [7]:
import lightgbm as lgb
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# 1. 生成 50000 个高维稀疏特征的模拟数据集，更贴近实际工业场景
X, y = make_classification(n_samples=50000, n_features=40, n_informative=30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. 将数据封装为 LightGBM 特有的 Dataset 格式（内部会直接进行直方图离散化）
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# 3. 配置参数
params = {
    'boosting_type': 'gbdt',
    'objective': 'binary',             # 二分类任务
    'metric': 'auc',                   # 评估指标
    'num_leaves': 31,                  # 叶子节点数（Leaf-wise的核心控制项，通常 2^(max_depth) - 1）
    'max_depth': 6,                    # 限制最大树深，防止过拟合
    'learning_rate': 0.05,             # 步长
    'feature_fraction': 0.8,           # 特征采样率（类似随机森林）
    'bagging_fraction': 0.8,           # 样本采样率
    'bagging_freq': 5,                 # 每 5 轮进行一次随机采样
    'verbose': -1                      # 减少不必要的日志输出
}

# 4. 模型训练 (加入 Early Stopping 机制防止过拟合)
# callbacks 列表里加入了 early_stopping 和 log 打印
callbacks = [
    lgb.early_stopping(stopping_rounds=10, verbose=True),
    lgb.log_evaluation(period=20)
]

model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[test_data],
    callbacks=callbacks
)

# 5. 预测与评估
# 注意：原生接口 predict 输出的是概率值值
pred_probs = model.predict(X_test, num_iteration=model.best_iteration)
preds = np.where(pred_probs > 0.5, 1, 0)

print("-" * 50)
print(f"【LightGBM】测试集最终 AUC: {roc_auc_score(y_test, pred_probs):.4f}")
print(f"【LightGBM】测试集最终准确率: {accuracy_score(y_test, preds):.4f}")

Training until validation scores don't improve for 10 rounds
[20]	valid_0's auc: 0.920514
[40]	valid_0's auc: 0.947386
[60]	valid_0's auc: 0.961082
[80]	valid_0's auc: 0.969872
[100]	valid_0's auc: 0.975529
[120]	valid_0's auc: 0.979509
[140]	valid_0's auc: 0.982272
[160]	valid_0's auc: 0.983895
[180]	valid_0's auc: 0.985364
[200]	valid_0's auc: 0.986632
[220]	valid_0's auc: 0.987698
[240]	valid_0's auc: 0.988298
[260]	valid_0's auc: 0.988863
[280]	valid_0's auc: 0.989475
[300]	valid_0's auc: 0.990124
[320]	valid_0's auc: 0.990474
[340]	valid_0's auc: 0.990846
[360]	valid_0's auc: 0.991101
[380]	valid_0's auc: 0.991368
[400]	valid_0's auc: 0.991512
[420]	valid_0's auc: 0.991726
[440]	valid_0's auc: 0.991903
[460]	valid_0's auc: 0.992122
[480]	valid_0's auc: 0.992264
[500]	valid_0's auc: 0.992485
Did not meet early stopping. Best iteration is:
[498]	valid_0's auc: 0.992499
--------------------------------------------------
【LightGBM】测试集最终 AUC: 0.9925
【LightGBM】测试集最终准确率: 0.9653
